In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--find-links",
    "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
    "arc-agi", "arcengine", "python-dotenv"])
print("deps OK")


In [ ]:
# Escribe el agente (vendor/my_agent_v47.py, ver atribución en el header)
AGENT_SOURCE = "# VENDORED (verbatim) from the public in-competition Kaggle notebook:\n#   poby7722/versrion-47-is-bset-score-0-54  (hidden LB 0.54, CPU-only exploration)\n# which itself ports techniques from Occam (github.com/g-baskin/occam, MIT) and the\n# MIT-licensed 3rd-place 'just-explore' solution. Kept byte-faithful as our proven\n# 0.54 baseline; our improvements go in subclasses/configs, never by editing this file.\n# See docs/DESIGN.md and paper/working_note_es.md for the strategy context.\n\n# ARC-AGI-3 explore2 MyAgent for Kaggle competition rerun.\n# Frontier graph exploration + learned counter mask + effective-action ordering.\n# Public-set config: default explore2 plus reorder disabled only for cn04.\nimport time\n\n\"\"\"Graph-based exploration agent for ARC-AGI-3.\n\nStrategy (no LLM, no GPU training - see PLAN.md):\n  * Treat each distinct frame as a node in a per-level state graph.\n  * Edge = (action) that transformed one frame into another.\n  * From the current node, try its untried actions first. When a node is\n    exhausted, BFS to the nearest node that still has untried actions and\n    replay the path to get there (\"frontier exploration\").\n  * For the complex click action (ACTION6) we do not enumerate all 64*64\n    coordinates blindly: we segment the frame into same-color connected\n    components and propose their centroids as click candidates, ordered by a\n    crude \"button-likeness\" heuristic (small, compact, non-background blobs).\n\nThis is intentionally deterministic and cheap. It is the v1 baseline we will\niterate on (world-model, object abstraction) per PLAN.md.\n\"\"\"\n\nimport logging\nfrom collections import deque\nfrom typing import Any, Optional\n\nfrom arcengine import FrameData, GameAction, GameState\n\nfrom ..agent import Agent\n\nlogger = logging.getLogger()\n\n\n# v3: width of the border band masked out before hashing a frame into a state\n# key. ARC-AGI-3 games render step counters / status bars along the edges; those\n# change every step and would otherwise explode the state space with spurious\n# nodes. Masking the border collapses them (technique ported from the MIT-\n# licensed 3rd-place \"just-explore\" solution, identify_status_bars_crude).\nSTATUS_BAR_BORDER = 3\n\n\ndef _mask_borders(grid: list, border: int = STATUS_BAR_BORDER) -> tuple:\n    \"\"\"Return a hashable copy of a 2D grid with the outer `border` cells zeroed,\n    so edge status bars don't create distinct states.\"\"\"\n    h = len(grid)\n    w = len(grid[0]) if h else 0\n    if h <= 2 * border or w <= 2 * border:\n        # too small to mask meaningfully; hash as-is\n        return tuple(tuple(row) for row in grid)\n    out = []\n    for r in range(h):\n        if r < border or r >= h - border:\n            out.append((0,) * w)\n        else:\n            row = grid[r]\n            out.append((0,) * border + tuple(row[border:w - border]) + (0,) * border)\n    return tuple(out)\n\n\ndef _grid_key(frame: Optional[list]) -> tuple:\n    \"\"\"Hashable signature of the frame, with edge status bars masked out (v3).\"\"\"\n    if not frame:\n        return ()\n    return tuple(_mask_borders(grid) for grid in frame)\n\n\ndef _last_grid(frame: Optional[list]) -> Optional[list]:\n    if not frame:\n        return None\n    return frame[-1]\n\n\ndef _connected_components(grid: list) -> list:\n    \"\"\"4-connectivity same-color components. Returns list of dicts with\n    color, size, bbox, and centroid (x=col, y=row).\"\"\"\n    h = len(grid)\n    w = len(grid[0]) if h else 0\n    seen = [[False] * w for _ in range(h)]\n    comps: list = []\n    for sr in range(h):\n        for sc in range(w):\n            if seen[sr][sc]:\n                continue\n            color = grid[sr][sc]\n            q = deque([(sr, sc)])\n            seen[sr][sc] = True\n            cells = []\n            while q:\n                r, c = q.popleft()\n                cells.append((r, c))\n                for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):\n                    nr, nc = r + dr, c + dc\n                    if 0 <= nr < h and 0 <= nc < w and not seen[nr][nc] and grid[nr][nc] == color:\n                        seen[nr][nc] = True\n                        q.append((nr, nc))\n            rows = [c[0] for c in cells]\n            cols = [c[1] for c in cells]\n            comps.append(\n                {\n                    \"color\": color,\n                    \"size\": len(cells),\n                    \"bbox\": (min(rows), min(cols), max(rows), max(cols)),\n                    \"centroid\": (sum(cols) // len(cells), sum(rows) // len(cells)),\n                }\n            )\n    return comps\n\n\ndef _background_color(grid: list) -> int:\n    \"\"\"Most common color is assumed to be background.\"\"\"\n    counts: dict = {}\n    for row in grid:\n        for v in row:\n            counts[v] = counts.get(v, 0) + 1\n    return max(counts, key=counts.get)\n\n\nimport os as _os\n\n# Env-tunable click breadth (defaults 64/8 = the verified public-set config).\n_CLICK_LIMIT = int(_os.getenv(\"ARC_E2_CLICK_LIMIT\", \"64\"))\n_CLICK_STEP = int(_os.getenv(\"ARC_E2_CLICK_STEP\", \"8\"))\n\n\ndef _click_candidates(grid: Optional[list], limit: int = _CLICK_LIMIT,\n                      grid_step: int = _CLICK_STEP) -> list:\n    \"\"\"Propose (x, y) click points for ACTION6, ordered by priority.\n\n    Single flat pool of non-background blob centroids, sorted by button-likeness\n    (compact + small ranks first), then a coarse grid sweep for coverage.\n\n    NOTE: an earlier v3 grouped blobs into 5 hard priority tiers (ported from\n    the top-3 solution's frame_segments_to_action_groups). Benchmarks showed it\n    REGRESSED: tn36 dropped 112->30 states and 1->0 levels (reproduced twice),\n    because hard tiers force-exhaust the \"button-like\" group before trying\n    interaction cells that don't match the button heuristic, so within a 600-step\n    budget the agent never reaches them. The flat ordering still surfaces small\n    rare-color buttons first (high likeness) without starving other cells, and\n    matched or beat tiers on every game. Kept flat by evidence.\n\n    Returns up to `limit` (x, y) = (col, row) points.\n    \"\"\"\n    if not grid:\n        return []\n    h = len(grid)\n    w = len(grid[0]) if h else 0\n    bg = _background_color(grid)\n\n    scored = []\n    for comp in _connected_components(grid):\n        if comp[\"color\"] == bg:\n            continue\n        r0, c0, r1, c1 = comp[\"bbox\"]\n        bbox_area = max(1, (r1 - r0 + 1) * (c1 - c0 + 1))\n        fill = comp[\"size\"] / bbox_area\n        likeness = fill / (1.0 + comp[\"size\"])  # compact + small = more button-like\n        scored.append((likeness, comp[\"centroid\"]))\n    scored.sort(key=lambda t: t[0], reverse=True)\n\n    out: list = []\n    seen = set()\n    for _, cxy in scored:\n        if cxy not in seen:\n            seen.add(cxy)\n            out.append(cxy)\n\n    # coarse grid sweep (offset by half-step to hit cell centres) for coverage\n    half = max(1, grid_step // 2)\n    for y in range(half, h, grid_step):\n        for x in range(half, w, grid_step):\n            p = (x, y)\n            if p not in seen:\n                seen.add(p)\n                out.append(p)\n\n    return out[:limit]\n\n\nclass Explore(Agent):\n    \"\"\"Frontier graph exploration agent.\"\"\"\n\n    # ls20 enforces an ~80-action episode budget server-side (GAME_OVER after),\n    # so a high cap mainly matters for games that allow longer episodes / many\n    # RESET-and-retry trajectories. Keep generous; the graph carries across resets.\n    MAX_ACTIONS = 600\n\n    # After this many resets with no newly-discovered state, treat the game as a\n    # deterministic dead-end for this agent (avoids the ft09 reset loop).\n    MAX_EXHAUSTED_RESETS = 8\n\n    def __init__(self, *args: Any, **kwargs: Any) -> None:\n        super().__init__(*args, **kwargs)\n        self._nodes: dict = {}\n        self._edges: dict = {}\n        self._level_seen_reset = -1\n        self._replay_queue: deque = deque()\n        self._last_key: Optional[tuple] = None\n        self._last_plan: Optional[Any] = None\n        self._exhausted_resets = 0\n\n    def is_done(self, frames: list, latest_frame: FrameData) -> bool:\n        done = latest_frame.state is GameState.WIN\n        # Lightweight diagnostics: log state changes + periodic heartbeat so we\n        # can see WIN/GAME_OVER transitions (action log lines don't show state).\n        st = getattr(latest_frame.state, \"name\", str(latest_frame.state))\n        if st != getattr(self, \"_last_logged_state\", None):\n            logger.info(\n                \"STATE -> %s (action %d, levels_completed=%d, nodes=%d)\",\n                st, self.action_counter, latest_frame.levels_completed, len(self._nodes),\n            )\n            self._last_logged_state = st\n        return done\n\n    @staticmethod\n    def _as_action(a: Any) -> GameAction:\n        \"\"\"available_actions may arrive as ints (pydantic-coerced) or enums.\"\"\"\n        if isinstance(a, GameAction):\n            return a\n        return GameAction.from_id(a)\n\n    def _state_key(self, frame: Optional[list]) -> tuple:\n        \"\"\"Hashable state signature for a frame. Default = static border mask.\n        Subclasses (e.g. Explore2) override to apply a learned counter mask.\"\"\"\n        return _grid_key(frame)\n\n    def _simple_actions(self, latest_frame: FrameData) -> list:\n        avail = getattr(latest_frame, \"available_actions\", None)\n        if avail:\n            acts = [self._as_action(a) for a in avail]\n            acts = [a for a in acts if a is not GameAction.RESET]\n        else:\n            acts = [a for a in GameAction if a is not GameAction.RESET]\n        return acts\n\n    def _build_plans(self, latest_frame: FrameData) -> list:\n        \"\"\"A plan is either a GameAction (simple) or ('CLICK', x, y) for ACTION6.\"\"\"\n        plans: list = []\n        avail = self._simple_actions(latest_frame)\n        for a in avail:\n            # ACTION6 is the complex click action; handled separately below.\n            # (GameAction enum members are shared/mutated by set_data, so\n            #  a.is_simple() can flip True after a click \u2014 exclude explicitly.)\n            if a is GameAction.ACTION6:\n                continue\n            if a.is_simple():\n                plans.append(a)\n        if any(a is GameAction.ACTION6 for a in avail):\n            grid = _last_grid(latest_frame.frame)\n            for (x, y) in _click_candidates(grid):\n                plans.append((\"CLICK\", x, y))\n        return plans\n\n    def _plan_to_action(self, plan: Any) -> GameAction:\n        if isinstance(plan, tuple) and plan and plan[0] == \"CLICK\":\n            action = GameAction.ACTION6\n            action.set_data({\"x\": plan[1], \"y\": plan[2]})\n            action.reasoning = {\"strategy\": \"explore-click\", \"x\": plan[1], \"y\": plan[2]}\n            return action\n        action = plan\n        action.reasoning = {\"strategy\": \"explore-simple\"}\n        return action\n\n    def _register(self, key: tuple, latest_frame: FrameData) -> None:\n        if key not in self._nodes:\n            self._nodes[key] = {\"untried\": self._build_plans(latest_frame)}\n            self._edges.setdefault(key, [])\n            # Discovering a new state means exploration is still productive --\n            # clear the stuck counter so reset-loop detection only fires when we\n            # truly stop finding anything new.\n            self._exhausted_resets = 0\n\n    def _bfs_to_frontier(self, start: tuple) -> Optional[list]:\n        \"\"\"Return plans leading from start to a node with untried plans.\"\"\"\n        q: deque = deque([(start, [])])\n        visited = {start}\n        while q:\n            node, path = q.popleft()\n            info = self._nodes.get(node)\n            if info and info[\"untried\"] and node != start:\n                return path\n            for plan, child in self._edges.get(node, []):\n                if child not in visited:\n                    visited.add(child)\n                    q.append((child, path + [plan]))\n        return None\n\n    def choose_action(self, frames: list, latest_frame: FrameData) -> GameAction:\n        state = latest_frame.state\n\n        if state in (GameState.NOT_PLAYED, GameState.GAME_OVER):\n            self._replay_queue.clear()\n            self._last_key = None\n            return GameAction.RESET\n\n        if latest_frame.levels_completed != self._level_seen_reset:\n            self._level_seen_reset = latest_frame.levels_completed\n\n        key = self._state_key(latest_frame.frame)\n        self._register(key, latest_frame)\n\n        if self._last_key is not None and self._last_plan is not None:\n            edges = self._edges.setdefault(self._last_key, [])\n            if not any(p == self._last_plan and c == key for p, c in edges):\n                edges.append((self._last_plan, key))\n\n        if self._replay_queue:\n            plan = self._replay_queue.popleft()\n            self._last_key, self._last_plan = key, plan\n            return self._plan_to_action(plan)\n\n        info = self._nodes[key]\n        if info[\"untried\"]:\n            plan = info[\"untried\"].pop(0)\n            self._last_key, self._last_plan = key, plan\n            return self._plan_to_action(plan)\n\n        path = self._bfs_to_frontier(key)\n        if path:\n            self._replay_queue = deque(path)\n            plan = self._replay_queue.popleft()\n            self._last_key, self._last_plan = key, plan\n            return self._plan_to_action(plan)\n\n        # Whole reachable graph exhausted. RESET is only useful if the episode\n        # actually ended (GAME_OVER) or to retry a non-deterministic level. For\n        # a deterministic NOT_FINISHED state, reset returns to the same explored\n        # start -> infinite loop (this is exactly what trapped ft09 in v1).\n        # Count these and stop fighting once it's clearly stuck.\n        self._replay_queue.clear()\n        self._last_key, self._last_plan = None, None\n        self._exhausted_resets += 1\n        if self._exhausted_resets == 1:\n            logger.info(\n                \"Graph exhausted at %d states; resetting to retry.\", len(self._nodes)\n            )\n        if self._exhausted_resets >= self.MAX_EXHAUSTED_RESETS:\n            # Genuinely stuck: deterministic dead-end, no new states across many\n            # resets. Keep returning RESET (harness/MAX_ACTIONS will end it) but\n            # don't pretend we're exploring.\n            if self._exhausted_resets == self.MAX_EXHAUSTED_RESETS:\n                logger.info(\n                    \"Stuck: %d resets with no new states (%d total) -- \"\n                    \"deterministic dead-end for this agent.\",\n                    self._exhausted_resets, len(self._nodes),\n                )\n        return GameAction.RESET\n\n\n\"\"\"Explore2 \u2014 graph exploration with two techniques ported from Occam.\n\nOccam (github.com/g-baskin/occam, MIT, \"Sean Donahoe\") uses the same reset-replay\ngraph-exploration core as our Explore but reaches 17/25 public games (57.6% RHAE,\nCPU-only, no LLM) where the identical core gets 14/25. Two of its upgrades are the\nnamed reasons, both pure-Python / no-training, reimplemented here in our own\narchitecture (the env API differs, so this is a port, not a copy):\n\n  1. AUTO-LEARNED COUNTER/ANIMATION MASK (Occam: detect_counter_mask). Our Explore\n     hard-masks a fixed 3-cell border to kill edge step-counters. But some games\n     render the counter / an animation *inside* the scene, so every frame hashes\n     unique and the state graph explodes -> BFS never goes deep. Occam ANDs the\n     pixels that change on every consecutive transition over ~3 actions and zeros\n     them before hashing. We do the robust online version: accumulate, over a\n     warmup window, the cells that change in >= MASK_CHANGE_RATIO of transitions,\n     then freeze that as the mask. Guarded: never mask more than MASK_MAX_FRACTION\n     of the interior (else we'd collapse real game state -> keep border-only).\n\n  2. EFFECTIVE-ACTION ORDERING (Occam: _discover_and_prune + action_effectiveness).\n     Occam probes each action once from reset and hard-drops the ones that don't\n     change the masked hash. We use the softer online form: track, per simple\n     action, P(it changed the masked state), and when expanding a fresh node try\n     historically-effective actions first (a disabled/no-op button sinks to the\n     back). Softer than a hard one-shot prune because an action that is a no-op in\n     one state may matter in another; ordering keeps it available but late.\n\nEverything else (frontier BFS replay, click candidates, reset-loop breaker) is\ninherited unchanged from Explore, so `explore` stays the A/B control. Offline\nunit tests in tests/smoke_explore2.py drive the mask learner and action ranker\nwith hand-built frame sequences of known ground truth (no network, no game sim).\n\"\"\"\nimport os\nfrom typing import Any, Optional\n\nfrom arcengine import FrameData, GameAction\n\n\n\nclass Explore2(Explore):\n    \"\"\"Explore + learned counter mask + effective-action ordering.\"\"\"\n\n    # Observe this many transitions before freezing the learned counter mask.\n    MASK_WARMUP = 12\n    # A cell is \"counter/animation\" if it changed in >= this fraction of the\n    # observed transitions during warmup. Env-overridable for sweeps.\n    MASK_CHANGE_RATIO = float(os.getenv(\"ARC_E2_MASK_RATIO\", \"0.8\"))\n    # Safety: never mask more than this fraction of interior cells. Env-overridable.\n    MASK_MAX_FRACTION = float(os.getenv(\"ARC_E2_MASK_MAXFRAC\", \"0.20\"))\n\n    # Ablation toggles (env-overridable) so we can A/B which component helps on a\n    # given game. Both default ON. ARC_E2_MASK=0 disables the counter mask;\n    # ARC_E2_REORDER=0 disables effective-action ordering. Used to diagnose the\n    # cn04 regression (explore2 lost a level explore had) without forking code.\n    USE_MASK = os.getenv(\"ARC_E2_MASK\", \"1\") != \"0\"\n    USE_REORDER = os.getenv(\"ARC_E2_REORDER\", \"1\") != \"0\"\n\n    def __init__(self, *args: Any, **kwargs: Any) -> None:\n        super().__init__(*args, **kwargs)\n        self._mask_cells: Optional[frozenset] = None  # frozen once learned\n        self._change_counts: dict = {}                # (r,c) -> times changed\n        self._transitions = 0\n        self._prev_interior: Optional[tuple] = None   # border-masked prev grid\n        self._prev_emitted: Optional[str] = None      # action name we last sent\n        # effectiveness: action name -> [n_changed, n_total]\n        self._effect: dict = {}\n\n    # -- learned mask -----------------------------------------------------\n    def _state_key(self, frame: Optional[list]) -> tuple:\n        if not frame:\n            return ()\n        out = []\n        for grid in frame:\n            m = _mask_borders(grid)  # tuple-of-tuples, border already zeroed\n            if self._mask_cells and self.USE_MASK:\n                rows = [list(r) for r in m]\n                h = len(rows)\n                w = len(rows[0]) if h else 0\n                for (r, c) in self._mask_cells:\n                    if 0 <= r < h and 0 <= c < w:\n                        rows[r][c] = 0\n                m = tuple(tuple(r) for r in rows)\n            out.append(m)\n        return tuple(out)\n\n    def _observe_transition(self, frame: Optional[list]) -> None:\n        \"\"\"Diff the new interior grid against the previous one; accumulate which\n        cells change and how effective the last emitted action was. Freeze the\n        counter mask once enough transitions are seen.\"\"\"\n        if not frame:\n            return\n        interior = _mask_borders(frame[-1])\n        prev = self._prev_interior\n        self._prev_interior = interior\n        if prev is None or len(prev) != len(interior):\n            return\n        h = len(interior)\n        w = len(interior[0]) if h else 0\n        changed_any = False\n        learning = self._mask_cells is None  # only scan cells while unfrozen\n        for r in range(h):\n            pr, cr = prev[r], interior[r]\n            if pr == cr:\n                continue\n            changed_any = True\n            if learning:\n                for c in range(w):\n                    if pr[c] != cr[c]:\n                        k = (r, c)\n                        self._change_counts[k] = self._change_counts.get(k, 0) + 1\n        self._transitions += 1\n        # attribute effectiveness to the action that produced this transition\n        if self._prev_emitted is not None:\n            e = self._effect.setdefault(self._prev_emitted, [0, 0])\n            e[1] += 1\n            if changed_any:\n                e[0] += 1\n        if learning and self._transitions >= self.MASK_WARMUP:\n            self._freeze_mask(h, w)\n\n    def _freeze_mask(self, h: int, w: int) -> None:\n        b = STATUS_BAR_BORDER\n        interior_area = max(1, (h - 2 * b) * (w - 2 * b))\n        threshold = self.MASK_CHANGE_RATIO * self._transitions\n        cells = {k for k, n in self._change_counts.items() if n >= threshold}\n        # guard: a too-large mask would erase real game state -> keep border-only\n        if len(cells) > self.MASK_MAX_FRACTION * interior_area:\n            cells = set()\n        self._mask_cells = frozenset(cells)\n        self._change_counts = {}  # free memory\n\n    # -- effective-action ordering ---------------------------------------\n    def _build_plans(self, latest_frame: FrameData) -> list:\n        plans = super()._build_plans(latest_frame)\n        if not self._effect or not self.USE_REORDER:\n            return plans\n\n        def rank(plan: Any) -> float:\n            name = getattr(plan, \"name\", str(plan))\n            n_chg, n_tot = self._effect.get(name, (0, 0))\n            # unknown actions -> 0.5 (worth exploring); known -> measured P(change)\n            return (n_chg / n_tot) if n_tot else 0.5\n\n        simple = [p for p in plans if not isinstance(p, tuple)]\n        clicks = [p for p in plans if isinstance(p, tuple)]\n        simple.sort(key=rank, reverse=True)  # most effective first\n        return simple + clicks               # clicks keep inherited likeness order\n\n    # -- hook into the loop ----------------------------------------------\n    def choose_action(self, frames: list, latest_frame: FrameData) -> GameAction:\n        # Learn from the transition the previous action produced, THEN decide so\n        # this step's state key already uses the up-to-date mask.\n        self._observe_transition(latest_frame.frame)\n        action = super().choose_action(frames, latest_frame)\n        self._prev_emitted = getattr(action, \"name\", str(action))\n        return action\n\n\nclass MyAgent(Explore2):\n    MAX_ACTIONS = 15000\n    RESET_LOOP_BREAK = 20\n    MAX_RUNTIME_SECONDS = 8 * 3600 - 300\n\n    DEFAULT_AGENT_CONFIG = {\n        \"USE_MASK\": True,\n        \"USE_REORDER\": True,\n        \"MASK_CHANGE_RATIO\": 0.8,\n        \"MASK_MAX_FRACTION\": 0.20,\n    }\n    GAME_CONFIG = {\n        \"cn04\": {\"USE_REORDER\": False},\n    }\n\n    def __init__(self, *args: Any, **kwargs: Any) -> None:\n        super().__init__(*args, **kwargs)\n        self.start_time = time.time()\n        self._consecutive_resets = 0\n        short = self.game_id.split(\"-\")[0]\n        for name, value in self.DEFAULT_AGENT_CONFIG.items():\n            setattr(self, name, value)\n        for name, value in self.GAME_CONFIG.get(short, {}).items():\n            setattr(self, name, value)\n\n    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:\n        if latest_frame.state is GameState.WIN:\n            return True\n        if self._consecutive_resets >= self.RESET_LOOP_BREAK:\n            return True\n        return (time.time() - self.start_time) >= self.MAX_RUNTIME_SECONDS\n\n    def choose_action(self, frames: list, latest_frame: FrameData) -> GameAction:\n        action = super().choose_action(frames, latest_frame)\n        if action.name == \"RESET\":\n            self._consecutive_resets += 1\n        else:\n            self._consecutive_resets = 0\n        return action\n"
with open('/kaggle/working/my_agent.py', 'w', encoding='utf-8') as f:
    f.write(AGENT_SOURCE)
print('my_agent.py escrito:', len(AGENT_SOURCE), 'chars')


In [ ]:
import os, shutil, subprocess
if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    subprocess.run(["curl", "--fail", "--retry", "999", "--retry-all-errors",
                    "--retry-delay", "5", "--retry-max-time", "600",
                    "http://gateway:8001/api/games"], check=True)
    shutil.copytree("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents",
                    "/kaggle/working/ARC-AGI-3-Agents", dirs_exist_ok=True)
    shutil.copy("/kaggle/working/my_agent.py",
                "/kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py")
    with open("/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py", "w") as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent
load_dotenv()
AVAILABLE_AGENTS: dict[str, Type[Agent]] = {"random": Random, "myagent": MyAgent}
""")
    with open("/kaggle/working/ARC-AGI-3-Agents/.env", "w") as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
RECORDINGS_DIR=/kaggle/working/server_recording
""")
    env = dict(os.environ, MPLBACKEND="agg")
    subprocess.run(["python", "main.py", "--agent", "myagent"],
                   cwd="/kaggle/working/ARC-AGI-3-Agents", env=env, check=False)
else:
    print("Save & Run: instalacion validada; el agente juega solo en el rerun (igual que el original 0.54)")


In [ ]:
import os
if not os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    import pandas as pd
    pd.DataFrame(data=[["1_0", "1", True, 1]],
                 columns=["row_id", "game_id", "end_of_game", "score"]).to_parquet(
        "/kaggle/working/submission.parquet", index=False)
    print("submission.parquet dummy escrito")
